# LAB2 — CRAG Completo com LangGraph StateGraph

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 8 — Self-RAG e CRAG: Recuperação Reflexiva e Adaptativa**

---

## Objetivo

Neste laboratório, implementamos o **CRAG** (Corrective Retrieval-Augmented Generation), uma arquitetura que adiciona um **mecanismo de correção** ao RAG tradicional, roteando automaticamente entre documentos locais e busca na web quando os documentos recuperados não são suficientemente relevantes.

### Diferença entre RAG, Self-RAG e CRAG

| Aspecto | RAG Tradicional | Self-RAG | CRAG |
|---------|----------------|----------|------|
| Busca documentos? | Sempre | Condicionalmente | Sempre |
| Avalia relevância? | Não | Sim (tokens) | Sim (LLM-as-Judge) |
| Fallback para web? | Não | Não | Sim |
| Usa grafo de estados? | Não | Não | Sim (LangGraph) |

### Arquitetura CRAG com LangGraph

```
                    ┌─────────────────────────────────┐
                    │         INICIO (query)          │
                    └─────────────┬───────────────────┘
                                  │
                                  ▼
                    ┌─────────────────────────────────┐
                    │   retrieve: ChromaDB k=3        │
                    └─────────────┬───────────────────┘
                                  │
                                  ▼
                    ┌─────────────────────────────────┐
                    │  grade_documents: LLM-as-Judge  │
                    │  gera relevance_scores [0,1]    │
                    └────────┬─────────────┬──────────┘
                             │             │
                   score≥0.5 │             │ score<0.5
                             ▼             ▼
              ┌──────────────────┐  ┌───────────────────────┐
              │    generate      │  │  web_search (Tavily)  │
              │  (docs locais)   │  │  busca online         │
              └────────┬─────────┘  └──────────┬────────────┘
                       │                       │
                       │            ┌──────────┘
                       │            ▼
                       │  ┌────────────────────┐
                       │  │  generate          │
                       │  │  (docs + web)      │
                       │  └────────┬───────────┘
                       │           │
                       └─────┬─────┘
                             ▼
                           [END]
```

### Componentes Principais
- **LangGraph StateGraph**: orquestrador de fluxo com estado compartilhado
- **ChromaDB**: índice vetorial local com documentos jurídicos
- **GPT-4o-mini**: avaliador de relevância (LLM-as-Judge) e gerador de respostas
- **Tavily**: motor de busca na web para documentos não encontrados localmente


---
## PASSO 1 — Instalação das Dependências

Instalamos as bibliotecas principais do ecossistema LangChain, LangGraph, ChromaDB e Tavily.


In [ ]:
# Instala todas as dependências necessárias para o CRAG
# -q suprime a saída verbosa do pip
!pip install langchain langgraph langchain-openai langchain-community chromadb sentence-transformers tavily-python -q

print("✅ Dependências instaladas com sucesso!")
print("   - langchain: framework principal")
print("   - langgraph: orquestrador de grafos de estado")
print("   - langchain-openai: integração com GPT-4o-mini")
print("   - langchain-community: integrações com ferramentas externas")
print("   - chromadb: banco vetorial local")
print("   - sentence-transformers: modelo de embeddings")
print("   - tavily-python: busca na web")

---
## PASSO 2 — Configurar Chaves de API

Configuramos as chaves de API necessárias:
- **OPENAI_API_KEY**: para usar GPT-4o-mini como avaliador e gerador
- **TAVILY_API_KEY**: para busca na web quando documentos locais são insuficientes

> **Dica**: No Google Colab, use `userdata.get()` para acessar secrets armazenados com segurança em `Ferramentas > Secrets`.


In [ ]:
import os

# ─── Configuração de API Keys ───────────────────────────────────────────────
# No Google Colab: configure em Ferramentas > Secrets
# Nome das secrets: OPENAI_API_KEY e TAVILY_API_KEY

try:
    # Tenta usar a API do Google Colab para acessar secrets
    from google.colab import userdata
    
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
    
    print("✅ Chaves carregadas via Google Colab Secrets")

except Exception:
    # Fallback: usa variáveis de ambiente (para rodar fora do Colab)
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'sua-chave-openai-aqui')
    TAVILY_API_KEY = os.environ.get('TAVILY_API_KEY', 'sua-chave-tavily-aqui')
    
    print("⚠️ Colab Secrets não disponível. Usando variáveis de ambiente.")
    print("   Configure OPENAI_API_KEY e TAVILY_API_KEY no ambiente.")

# Define as chaves como variáveis de ambiente (exigido pelas bibliotecas)
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

# Verifica se as chaves foram configuradas (mostra apenas os primeiros e últimos caracteres)
def verificar_chave(nome: str, chave: str) -> None:
    if chave and chave != f'sua-chave-{nome.lower().replace("_", "-")}-aqui' and len(chave) > 10:
        print(f"   {nome}: {chave[:8]}...{chave[-4:]} ✅")
    else:
        print(f"   {nome}: NÃO CONFIGURADA ❌")

print("\n🔑 Status das chaves:")
verificar_chave('OPENAI_API_KEY', OPENAI_API_KEY)
verificar_chave('TAVILY_API_KEY', TAVILY_API_KEY)

---
## PASSO 3 — Definir o GraphState

O **GraphState** é o contêiner de estado compartilhado entre todos os nós do grafo LangGraph. Cada nó lê e escreve neste estado, permitindo que informações fluam pelo pipeline de forma organizada.

Usamos `TypedDict` para garantir tipagem estática e documentação clara de cada campo.


In [ ]:
from typing import TypedDict, List, Optional, Dict, Any
from langchain_core.documents import Document

class GraphState(TypedDict):
    """
    Estado compartilhado do grafo CRAG.
    
    Este dicionário é passado entre todos os nós do LangGraph,
    acumulando informações ao longo da execução do pipeline.
    
    Campos:
        question: A pergunta original do usuário (imutável)
        documents: Lista de documentos recuperados do ChromaDB
        generation: Resposta final gerada pelo LLM
        web_results: Resultados da busca web via Tavily (quando ativada)
        relevance_scores: Lista de scores de relevância [0.0-1.0] para cada documento
        route_taken: Rota tomada pelo pipeline ("local" ou "web")
    """
    question: str                           # Pergunta do usuário
    documents: List[Document]               # Documentos recuperados localmente
    generation: str                         # Resposta gerada pelo LLM
    web_results: Optional[List[Dict]]       # Resultados da busca web
    relevance_scores: Optional[List[float]] # Scores de relevância dos documentos
    route_taken: Optional[str]              # "local" ou "web"


# Demonstra a estrutura do estado com um exemplo
exemplo_estado: GraphState = {
    "question": "Qual o prazo prescricional para reparação civil?",
    "documents": [],           # Será preenchido pelo nó 'retrieve'
    "generation": "",          # Será preenchido pelo nó 'generate'
    "web_results": None,       # Será preenchido pelo nó 'web_search' (se ativado)
    "relevance_scores": None,  # Será preenchido pelo nó 'grade_documents'
    "route_taken": None        # Será definido pela função 'decide_route'
}

print("✅ GraphState definido com sucesso!")
print("\n📋 Campos do estado:")
print("   question       → pergunta original do usuário")
print("   documents      → documentos recuperados do ChromaDB")
print("   generation     → resposta final gerada pelo LLM")
print("   web_results    → resultados da busca Tavily (opcional)")
print("   relevance_scores → scores de relevância [0.0-1.0] por documento")
print("   route_taken    → 'local' ou 'web' (rastreamento da rota)")

---
## PASSO 4 — Criar Índice ChromaDB com Corpus Jurídico

Construímos o índice vetorial com **10 documentos jurídicos** abrangendo as principais áreas do direito brasileiro: direito civil, penal, constitucional, administrativo e especial.


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document

# ─── Corpus Jurídico Hardcoded ──────────────────────────────────────────────
# 10 documentos cobrindo diferentes áreas do direito brasileiro
CORPUS_JURIDICO = [
    {
        "id": "cc_186",
        "texto": "Art. 186. Aquele que, por ação ou omissão voluntária, negligência ou imprudência, violar direito e causar dano a outrem, ainda que exclusivamente moral, comete ato ilícito.",
        "titulo": "CC - Art. 186 - Ato Ilícito",
        "fonte": "Lei nº 10.406/2002 - Código Civil",
        "categoria": "responsabilidade_civil"
    },
    {
        "id": "cc_206",
        "texto": "Art. 206. Prescreve: §3º Em três anos: V - a pretensão de reparação civil. §1º Em um ano: I - a pretensão dos hospedeiros ou fornecedores de víveres destinados a consumo no próprio estabelecimento, para o pagamento da hospedagem ou dos alimentos.",
        "titulo": "CC - Art. 206 - Prazos Prescricionais",
        "fonte": "Lei nº 10.406/2002 - Código Civil",
        "categoria": "prescricao"
    },
    {
        "id": "cpp_302",
        "texto": "Art. 302. Considera-se em flagrante delito quem: I - está cometendo a infração penal; II - acaba de cometê-la; III - é perseguido, logo após, pela autoridade, pelo ofendido ou por qualquer pessoa, em situação que faça presumir ser autor da infração; IV - é encontrado, logo depois, com instrumentos, armas, objetos ou papéis que façam presumir ser ele autor da infração.",
        "titulo": "CPP - Art. 302 - Flagrante Delito",
        "fonte": "Decreto-Lei nº 3.689/1941 - Código de Processo Penal",
        "categoria": "processo_penal"
    },
    {
        "id": "cf_5",
        "texto": "Art. 5º Todos são iguais perante a lei, sem distinção de qualquer natureza, garantindo-se aos brasileiros e aos estrangeiros residentes no País a inviolabilidade do direito à vida, à liberdade, à igualdade, à segurança e à propriedade. II - ninguém será obrigado a fazer ou deixar de fazer alguma coisa senão em virtude de lei; XXXIX - não há crime sem lei anterior que o defina, nem pena sem prévia cominação legal.",
        "titulo": "CF/88 - Art. 5º - Princípio da Legalidade e Direitos Fundamentais",
        "fonte": "Constituição da República Federativa do Brasil de 1988",
        "categoria": "direitos_fundamentais"
    },
    {
        "id": "cp_121",
        "texto": "Art. 121. Matar alguém: Pena - reclusão, de seis a vinte anos. Homicídio qualificado: §2º Se o homicídio é cometido: I - mediante paga ou promessa de recompensa, ou por outro motivo torpe; II - por motivo fútil; III - com emprego de veneno, fogo, explosivo, asfixia, tortura ou outro meio insidioso ou cruel; IV - à traição, de emboscada, ou mediante dissimulação; V - para assegurar a execução, a ocultação, a impunidade ou vantagem de outro crime: Pena - reclusão, de doze a trinta anos.",
        "titulo": "CP - Art. 121 - Homicídio Simples e Qualificado",
        "fonte": "Decreto-Lei nº 2.848/1940 - Código Penal",
        "categoria": "direito_penal"
    },
    {
        "id": "lmp_5",
        "texto": "Art. 5º Para os efeitos desta Lei, configura violência doméstica e familiar contra a mulher qualquer ação ou omissão baseada no gênero que lhe cause morte, lesão, sofrimento físico, sexual ou psicológico e dano moral ou patrimonial: I - no âmbito da unidade doméstica; II - no âmbito da família; III - em qualquer relação íntima de afeto, na qual o agressor conviva ou tenha convivido com a ofendida, independentemente de coabitação. Parágrafo único. As relações pessoais enunciadas neste artigo independem de orientação sexual.",
        "titulo": "Lei Maria da Penha - Art. 5º - Violência Doméstica",
        "fonte": "Lei nº 11.340/2006 - Lei Maria da Penha",
        "categoria": "violencia_domestica"
    },
    {
        "id": "lia_9",
        "texto": "Art. 9º Constitui ato de improbidade administrativa importando enriquecimento ilícito auferir qualquer tipo de vantagem patrimonial indevida em razão do exercício de cargo, mandato, função, emprego ou atividade nas entidades mencionadas no art. 1° desta lei, e notadamente: I - receber, para si ou para outrem, dinheiro, bem móvel ou imóvel, ou qualquer outra vantagem econômica, direta ou indireta, a título de comissão, percentagem, gratificação ou presente de quem tenha interesse, direto ou indireto, que possa ser atingido ou amparado por ação ou omissão decorrente das atribuições do agente público.",
        "titulo": "Lei de Improbidade Administrativa - Art. 9º - Enriquecimento Ilícito",
        "fonte": "Lei nº 8.429/1992 - Lei de Improbidade Administrativa",
        "categoria": "improbidade_administrativa"
    },
    {
        "id": "eca_112",
        "texto": "Art. 112. Verificada a prática de ato infracional, a autoridade competente poderá aplicar ao adolescente as seguintes medidas: I - advertência; II - obrigação de reparar o dano; III - prestação de serviços à comunidade; IV - liberdade assistida; V - inserção em regime de semi-liberdade; VI - internação em estabelecimento educacional; VII - qualquer uma das previstas no art. 101, I a VI. §1º A medida aplicada ao adolescente levará em conta a sua capacidade de cumpri-la, as circunstâncias e a gravidade da infração.",
        "titulo": "ECA - Art. 112 - Medidas Socioeducativas",
        "fonte": "Lei nº 8.069/1990 - Estatuto da Criança e do Adolescente",
        "categoria": "eca"
    },
    {
        "id": "lac_2",
        "texto": "Art. 2º As pessoas jurídicas serão responsabilizadas objetivamente, nos âmbitos administrativo e civil, pelos atos lesivos previstos nesta Lei praticados em seu interesse ou benefício, exclusivo ou não. Art. 3º A responsabilização da pessoa jurídica não exclui a responsabilidade individual de seus dirigentes ou administradores ou de qualquer pessoa natural, autora, coautora ou partícipe do ato ilícito.",
        "titulo": "Lei Anticorrupção - Arts. 2º e 3º - Responsabilidade das Pessoas Jurídicas",
        "fonte": "Lei nº 12.846/2013 - Lei Anticorrupção",
        "categoria": "anticorrupcao"
    },
    {
        "id": "stj_s227",
        "texto": "Súmula 227. A pessoa jurídica pode sofrer dano moral. Precedentes: REsp 134.993-MA, REsp 4.236-SP, REsp 52.842-RS. A honra objetiva da pessoa jurídica — sua reputação, credibilidade e imagem perante a sociedade — pode ser ofendida, gerando direito à indenização por danos morais.",
        "titulo": "STJ - Súmula 227 - Dano Moral da Pessoa Jurídica",
        "fonte": "Superior Tribunal de Justiça - Súmula nº 227",
        "categoria": "dano_moral"
    }
]

# ─── Inicialização do ChromaDB e Embeddings ─────────────────────────────────
print("Carregando modelo de embeddings...")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modelo de embeddings carregado!")

# Cria cliente ChromaDB em memória
chroma_client = chromadb.Client()

# Cria (ou recria) a coleção
try:
    chroma_client.delete_collection("crag_juridico")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="crag_juridico",
    metadata={"hnsw:space": "cosine"}  # Métrica de similaridade cosseno
)

# Gera embeddings e indexa os documentos
print("\nIndexando documentos no ChromaDB...")
textos = [doc['texto'] for doc in CORPUS_JURIDICO]
ids = [doc['id'] for doc in CORPUS_JURIDICO]
metadados = [{"titulo": doc['titulo'], "fonte": doc['fonte'], "categoria": doc['categoria']} for doc in CORPUS_JURIDICO]

embeddings = embedding_model.encode(textos, show_progress_bar=True)

collection.add(
    documents=textos,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadados
)

print(f"\n✅ {len(CORPUS_JURIDICO)} documentos indexados no ChromaDB!")
print("\n📚 Documentos disponíveis:")
for doc in CORPUS_JURIDICO:
    print(f"   [{doc['categoria']}] {doc['titulo']}")

---
## PASSO 5 — Implementar os 4 Nós do Grafo

Cada **nó** do LangGraph é uma função que:
1. Recebe o estado atual (`GraphState`)
2. Executa uma operação específica
3. Retorna um dicionário com as atualizações do estado

Os 4 nós implementados são:
- `retrieve`: busca documentos no ChromaDB
- `grade_documents`: avalia relevância com LLM-as-Judge
- `web_search`: busca na web via Tavily
- `generate`: gera a resposta final


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# Inicializa o modelo de linguagem
# GPT-4o-mini: boa relação custo-benefício para avaliação e geração
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,  # Temperature zero para avaliações determinísticas
    api_key=OPENAI_API_KEY
)

print("✅ LLM GPT-4o-mini configurado!")

# ─── NÓ 1: retrieve ─────────────────────────────────────────────────────────
def retrieve(state: GraphState) -> GraphState:
    """
    Nó de recuperação: busca documentos relevantes no ChromaDB.
    
    Gera o embedding da pergunta e busca os 3 documentos mais
    semanticamente próximos no índice local.
    
    Args:
        state: Estado atual do grafo com a pergunta do usuário
    
    Returns:
        Atualização do estado com a lista de documentos recuperados
    """
    print("\n[Nó: retrieve] Buscando documentos no ChromaDB...")
    
    question = state["question"]
    
    # Gera embedding da pergunta
    query_embedding = embedding_model.encode([question])[0]
    
    # Busca os 3 documentos mais próximos (k=3)
    resultados = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=3,
        include=['documents', 'metadatas', 'distances']
    )
    
    # Converte para objetos Document do LangChain
    # Document é a estrutura padrão usada em todo o ecossistema LangChain
    documents = [
        Document(
            page_content=resultados['documents'][0][i],
            metadata={
                **resultados['metadatas'][0][i],
                # Converte distância cosseno em score de similaridade 0-1
                'similarity': round(1 - resultados['distances'][0][i], 4)
            }
        )
        for i in range(len(resultados['documents'][0]))
    ]
    
    print(f"   Recuperados {len(documents)} documentos:")
    for doc in documents:
        print(f"     [{doc.metadata['similarity']:.2f}] {doc.metadata['titulo']}")
    
    # Retorna apenas as atualizações do estado (não precisa retornar tudo)
    return {"documents": documents}


print("✅ Nó 'retrieve' implementado!")

In [ ]:
# ─── NÓ 2: grade_documents ──────────────────────────────────────────────────

# Prompt para o avaliador LLM-as-Judge
# O modelo avalia cada documento e retorna um score numérico de 0 a 1
PROMPT_GRADER = ChatPromptTemplate.from_messages([
    ("system", """Você é um avaliador especializado em relevância de documentos jurídicos.
Sua tarefa é avaliar se um documento jurídico é relevante para responder a uma pergunta.

Retorne APENAS um número decimal entre 0.0 e 1.0:
- 1.0 = totalmente relevante (o documento responde diretamente à pergunta)
- 0.7 = bastante relevante (o documento contém informação útil relacionada)
- 0.5 = moderadamente relevante (relação parcial com o tema)
- 0.3 = pouco relevante (menciona o tema superficialmente)
- 0.0 = irrelevante (sem relação com a pergunta)

RETORNE APENAS O NÚMERO, sem texto adicional. Exemplo: 0.85"""),
    ("human", """Pergunta: {question}

Documento:
Título: {titulo}
Conteúdo: {content}

Score de relevância (0.0 a 1.0):""")
])


def grade_documents(state: GraphState) -> GraphState:
    """
    Nó de avaliação: usa LLM-as-Judge para pontuar a relevância de cada documento.
    
    O LLM avalia cada documento recuperado e atribui um score de 0 a 1
    indicando quão relevante ele é para responder a pergunta.
    
    Esta é a etapa 'Corrective' do CRAG: se os scores forem baixos,
    o sistema decide buscar na web em vez de usar os docs locais.
    
    Args:
        state: Estado com pergunta e lista de documentos
    
    Returns:
        Atualização com relevance_scores (lista de floats)
    """
    print("\n[Nó: grade_documents] Avaliando relevância dos documentos...")
    
    question = state["question"]
    documents = state["documents"]
    scores = []
    
    for i, doc in enumerate(documents):
        try:
            # Invoca o LLM avaliador para cada documento
            resposta = llm.invoke(
                PROMPT_GRADER.format_messages(
                    question=question,
                    titulo=doc.metadata.get('titulo', 'Sem título'),
                    content=doc.page_content
                )
            )
            
            # Extrai o score numérico da resposta do LLM
            # Usa regex para capturar o número mesmo se houver texto extra
            import re
            numeros = re.findall(r'\d+\.?\d*', resposta.content.strip())
            
            if numeros:
                score = float(numeros[0])
                # Garante que o score está dentro do intervalo [0, 1]
                score = max(0.0, min(1.0, score))
            else:
                # Se não conseguir extrair o número, assume score médio
                score = 0.5
                
        except Exception as e:
            print(f"   ⚠️ Erro na avaliação do documento {i+1}: {e}")
            score = 0.5  # Score padrão em caso de erro
        
        scores.append(score)
        print(f"   Doc {i+1} [{doc.metadata.get('titulo', '?')[:40]}]: score={score:.2f}")
    
    # Score médio como indicador geral de qualidade dos documentos
    score_medio = sum(scores) / len(scores) if scores else 0.0
    print(f"   Score médio: {score_medio:.2f} (threshold=0.5)")
    
    return {"relevance_scores": scores}


print("✅ Nó 'grade_documents' implementado!")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# ─── NÓ 3: web_search ───────────────────────────────────────────────────────

# Inicializa a ferramenta de busca Tavily
# max_results=3: busca até 3 resultados da web
tavily_search = TavilySearchResults(
    max_results=3,
    api_key=TAVILY_API_KEY
)


def web_search(state: GraphState) -> GraphState:
    """
    Nó de busca web: ativado quando documentos locais são insuficientes.
    
    Usa a API Tavily para buscar informações recentes na web,
    especialmente útil para:
    - Decisões jurídicas recentes (2024/2025)
    - Alterações legislativas
    - Jurisprudência mais atual
    
    Args:
        state: Estado com a pergunta do usuário
    
    Returns:
        Atualização com web_results e route_taken="web"
    """
    print("\n[Nó: web_search] Buscando na web via Tavily...")
    
    question = state["question"]
    
    # Enriquece a query com contexto jurídico para melhores resultados
    query_enriquecida = f"direito brasileiro jurisprudência {question}"
    
    try:
        # Executa a busca na web
        resultados = tavily_search.invoke(query_enriquecida)
        
        print(f"   Encontrados {len(resultados)} resultados na web:")
        for i, res in enumerate(resultados, 1):
            url = res.get('url', 'N/A')
            titulo = res.get('title', 'Sem título')
            print(f"   {i}. {titulo[:60]}")
            print(f"      URL: {url[:70]}")
        
    except Exception as e:
        print(f"   ⚠️ Erro na busca web: {e}")
        # Cria resultado simulado para não quebrar o pipeline em ambiente sem API
        resultados = [
            {
                "title": "Resultado simulado (API Tavily não disponível)",
                "content": f"Busca web simulada para: {question}. Configure a TAVILY_API_KEY para resultados reais.",
                "url": "https://www.tavily.com"
            }
        ]
    
    # Define route_taken para rastreamento
    return {
        "web_results": resultados,
        "route_taken": "web"
    }


print("✅ Nó 'web_search' implementado!")

In [ ]:
# ─── NÓ 4: generate ─────────────────────────────────────────────────────────

# Prompt para geração de resposta com documentos locais
PROMPT_GENERATE_LOCAL = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente jurídico especializado em direito brasileiro.
Responda à pergunta do usuário de forma clara, precisa e fundamentada nos documentos fornecidos.
Cite sempre a fonte legal (artigo, lei, súmula) quando relevante.
Responda em português de forma profissional."""),
    ("human", """Pergunta: {question}

Documentos de referência:
{context}

Resposta fundamentada:""")
])

# Prompt para geração com documentos locais + resultados web
PROMPT_GENERATE_WEB = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente jurídico especializado em direito brasileiro.
Os documentos locais foram insuficientes para responder a pergunta, então complementamos com busca na web.
Combine as fontes disponíveis para dar a melhor resposta possível.
Responda em português de forma profissional, indicando as fontes usadas."""),
    ("human", """Pergunta: {question}

Documentos locais (relevância baixa, mas podem ter contexto útil):
{context_local}

Resultados da busca web:
{context_web}

Resposta combinada:""")
])


def generate(state: GraphState) -> GraphState:
    """
    Nó de geração: produz a resposta final combinando documentos disponíveis.
    
    Comportamento:
    - Se web_results está disponível: usa docs locais + web para gerar resposta
    - Se apenas docs locais: usa somente eles
    
    Args:
        state: Estado com pergunta, documentos e (opcionalmente) resultados web
    
    Returns:
        Atualização com generation (texto da resposta) e route_taken
    """
    print("\n[Nó: generate] Gerando resposta final...")
    
    question = state["question"]
    documents = state["documents"]
    web_results = state.get("web_results")
    
    # Formata os documentos locais para o prompt
    context_local = "\n\n".join([
        f"[{doc.metadata.get('titulo', 'Documento')}]\n{doc.page_content}"
        for doc in documents
    ])
    
    if web_results:
        # ── GERAÇÃO COM WEB: combina docs locais e resultados web ────────────
        print("   Modo: documentos locais + resultados web")
        
        # Formata os resultados da web para o prompt
        context_web = "\n\n".join([
            f"[{res.get('title', 'Web Result')}]\n{res.get('content', '')[:300]}"
            for res in web_results
        ])
        
        # Gera a resposta com contexto completo (local + web)
        resposta = llm.invoke(
            PROMPT_GENERATE_WEB.format_messages(
                question=question,
                context_local=context_local,
                context_web=context_web
            )
        )
        route = "web"
    else:
        # ── GERAÇÃO LOCAL: usa apenas documentos do ChromaDB ─────────────────
        print("   Modo: documentos locais apenas")
        
        # Gera a resposta apenas com contexto local
        resposta = llm.invoke(
            PROMPT_GENERATE_LOCAL.format_messages(
                question=question,
                context=context_local
            )
        )
        route = "local"
    
    # Exibe prévia da resposta
    texto_resposta = resposta.content
    preview = texto_resposta[:150] + "..." if len(texto_resposta) > 150 else texto_resposta
    print(f"   Resposta gerada ({len(texto_resposta)} chars): {preview}")
    
    return {
        "generation": texto_resposta,
        "route_taken": route
    }


print("✅ Nó 'generate' implementado!")
print("\n📋 Resumo dos 4 nós:")
print("   1. retrieve         → busca k=3 docs no ChromaDB")
print("   2. grade_documents  → LLM-as-Judge avalia relevância [0,1]")
print("   3. web_search       → busca Tavily max_results=3")
print("   4. generate         → GPT-4o-mini gera resposta final")

---
## PASSO 6 — Função de Roteamento Condicional

A função `decide_route` é o **cérebro do CRAG**: ela analisa os scores de relevância e decide se o pipeline deve:
- **Gerar diretamente** com os documentos locais (score médio ≥ 0.5)
- **Buscar na web** primeiro, para complementar com informação mais relevante (score médio < 0.5)


In [ ]:
# Threshold de relevância: se o score médio for abaixo disso,
# o sistema buscará na web para complementar a resposta
THRESHOLD_RELEVANCIA = 0.5


def decide_route(state: GraphState) -> str:
    """
    Função de roteamento condicional: decide o próximo nó baseado nos scores.
    
    Esta função é usada como edge condicional no LangGraph.
    Ela NÃO modifica o estado — apenas retorna uma string indicando
    qual nó deve ser executado a seguir.
    
    Lógica:
        - score_médio >= 0.5 → vai para 'generate' (docs locais são bons)
        - score_médio < 0.5  → vai para 'web_search' (precisa de mais informação)
    
    Args:
        state: Estado com relevance_scores calculados pelo grade_documents
    
    Returns:
        'generate' ou 'web_search'
    """
    scores = state.get("relevance_scores", [])
    
    if not scores:
        # Se não há scores (erro), vai para web como fallback seguro
        print("\n[Roteamento] ⚠️ Sem scores disponíveis. Roteando para web_search.")
        return "web_search"
    
    # Calcula a média dos scores de relevância
    score_medio = sum(scores) / len(scores)
    
    print(f"\n[Roteamento] Score médio: {score_medio:.3f} (threshold: {THRESHOLD_RELEVANCIA})")
    
    if score_medio >= THRESHOLD_RELEVANCIA:
        print(f"[Roteamento] ✅ Score acima do threshold → ROTA LOCAL (generate)")
        return "generate"
    else:
        print(f"[Roteamento] 🌐 Score abaixo do threshold → ROTA WEB (web_search)")
        return "web_search"


# Demonstra o funcionamento da função de roteamento com exemplos
print("🧪 Testando função de roteamento:")

# Exemplo 1: scores altos → rota local
estado_teste_1 = {"relevance_scores": [0.9, 0.85, 0.7]}
rota_1 = decide_route(estado_teste_1)
print(f"   Scores [0.9, 0.85, 0.7] → Rota: '{rota_1}'")

# Exemplo 2: scores baixos → rota web
estado_teste_2 = {"relevance_scores": [0.2, 0.15, 0.3]}
rota_2 = decide_route(estado_teste_2)
print(f"   Scores [0.2, 0.15, 0.3] → Rota: '{rota_2}'")

print("\n✅ Função de roteamento implementada e testada!")

---
## PASSO 7 — Construir e Compilar o Grafo LangGraph

Aqui montamos o grafo completo usando o `StateGraph` do LangGraph:
1. Adicionamos os 4 nós
2. Definimos o ponto de entrada
3. Adicionamos arestas (fixas e condicionais)
4. Compilamos o grafo em um `Runnable` executável


In [ ]:
from langgraph.graph import StateGraph, END

# ─── Construção do Grafo ─────────────────────────────────────────────────────

# Inicializa o StateGraph com o tipo de estado definido anteriormente
workflow = StateGraph(GraphState)

# ── Adição dos Nós ───────────────────────────────────────────────────────────
# Cada nó é registrado com um nome e uma função correspondente
workflow.add_node("retrieve", retrieve)               # Nó 1: recuperação
workflow.add_node("grade_documents", grade_documents)  # Nó 2: avaliação
workflow.add_node("web_search", web_search)            # Nó 3: busca web
workflow.add_node("generate", generate)                # Nó 4: geração

# ── Ponto de Entrada ─────────────────────────────────────────────────────────
# O pipeline sempre começa pelo nó 'retrieve'
workflow.set_entry_point("retrieve")

# ── Arestas Fixas (sempre executadas) ────────────────────────────────────────
# retrieve → grade_documents: após recuperar, sempre avalia relevância
workflow.add_edge("retrieve", "grade_documents")

# web_search → generate: após buscar na web, sempre gera resposta
workflow.add_edge("web_search", "generate")

# generate → END: após gerar, o pipeline termina
workflow.add_edge("generate", END)

# ── Aresta Condicional ───────────────────────────────────────────────────────
# grade_documents → decide_route → (generate OU web_search)
# A função decide_route retorna a string com o nome do próximo nó
workflow.add_conditional_edges(
    source="grade_documents",         # Nó de origem
    path=decide_route,                # Função que decide o destino
    path_map={                        # Mapeamento das decisões para nós
        "generate": "generate",       # Score alto → vai para generate
        "web_search": "web_search"     # Score baixo → vai para web_search
    }
)

# ── Compilação ───────────────────────────────────────────────────────────────
# Compila o grafo em um objeto executável (Runnable do LangChain)
app = workflow.compile()

print("✅ Grafo CRAG compilado com sucesso!")
print("\n📐 Estrutura do grafo:")
print("   retrieve → grade_documents")
print("   grade_documents → [CONDICIONAL]")
print("     ├── score≥0.5 → generate → END")
print("     └── score<0.5 → web_search → generate → END")

---
## PASSO 8 — Visualizar o Grafo

O LangGraph pode gerar uma visualização do grafo usando Mermaid, uma linguagem de diagramas. Isso facilita o entendimento da arquitetura do pipeline.


In [ ]:
from IPython.display import display, Image

# Tenta visualizar o grafo como imagem PNG usando Mermaid
# O try/except garante que o notebook não quebre se a visualização falhar
try:
    print("Gerando visualização do grafo...")
    
    # draw_mermaid_png() gera o diagrama do grafo como bytes de uma imagem PNG
    grafo_png = app.get_graph().draw_mermaid_png()
    
    print("✅ Visualização gerada com sucesso!")
    
    # Exibe a imagem no notebook
    display(Image(grafo_png))
    
except Exception as e:
    # Fallback: exibe representação textual do grafo
    print(f"⚠️ Não foi possível gerar imagem do grafo: {e}")
    print("\nRepresentação textual do grafo CRAG:")
    print("""
    ┌─────────────────────────────────┐
    │           __start__             │
    └──────────────┬──────────────────┘
                   │
                   ▼
    ┌─────────────────────────────────┐
    │           retrieve              │
    │     (ChromaDB k=3 docs)         │
    └──────────────┬──────────────────┘
                   │
                   ▼
    ┌─────────────────────────────────┐
    │        grade_documents          │
    │     (LLM-as-Judge scores)       │
    └────────┬─────────────┬──────────┘
             │             │
   score≥0.5 │             │ score<0.5
             ▼             ▼
    ┌──────────────┐  ┌───────────────┐
    │   generate   │  │  web_search   │
    │  (local)     │  │  (Tavily)     │
    └──────┬───────┘  └───────┬───────┘
           │                  │
           │         ┌────────┘
           │         ▼
           │  ┌──────────────┐
           │  │   generate   │
           │  │ (local+web)  │
           │  └──────┬───────┘
           │         │
           └────┬────┘
                ▼
    ┌─────────────────────────────────┐
    │            __end__              │
    └─────────────────────────────────┘
    """)

# Exibe também a representação Mermaid em texto para referência
try:
    print("\n📝 Código Mermaid do grafo:")
    print(app.get_graph().draw_mermaid())
except Exception as e:
    print(f"⚠️ Não foi possível gerar Mermaid: {e}")

---
## PASSO 9 — Testar com 5 Queries

Testamos o pipeline CRAG com 5 queries estrategicamente escolhidas:

**Queries que devem usar a ROTA LOCAL** (documentos no ChromaDB):
1. Art. 186 do CC — Ato ilícito (doc disponível localmente)
2. Art. 302 do CPP — Flagrante delito (doc disponível localmente)
3. Prazos prescricionais do CC — Art. 206 (doc disponível localmente)

**Queries que devem usar a ROTA WEB** (informações recentes não estão no corpus):
4. Decisões do STF de 2025 sobre privacy de dados
5. Alterações na legislação de segurança pública em 2024


In [ ]:
import time

# Define as 5 queries de teste com suas expectativas de rota
QUERIES_CRAG = [
    {
        "question": "O que é ato ilícito segundo o Código Civil brasileiro?",
        "rota_esperada": "local",
        "descricao": "Art. 186 CC — documento disponível localmente"
    },
    {
        "question": "Quais são as hipóteses de flagrante delito no CPP?",
        "rota_esperada": "local",
        "descricao": "Art. 302 CPP — documento disponível localmente"
    },
    {
        "question": "Qual o prazo prescricional para pretensão de reparação civil no Código Civil?",
        "rota_esperada": "local",
        "descricao": "Art. 206 CC — documento disponível localmente"
    },
    {
        "question": "Quais foram as principais decisões do STF sobre proteção de dados pessoais em 2024 e 2025?",
        "rota_esperada": "web",
        "descricao": "Jurisprudência recente 2024/2025 — não está no corpus local"
    },
    {
        "question": "Quais foram as mudanças na legislação de segurança pública brasileira aprovadas em 2024?",
        "rota_esperada": "web",
        "descricao": "Legislação recente 2024 — requer busca atualizada na web"
    }
]

# Armazena todos os resultados para análise posterior
resultados_crag = []

# Executa o pipeline para cada query
for i, query_info in enumerate(QUERIES_CRAG, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}/5: {query_info['descricao']}")
    print(f"Rota esperada: {query_info['rota_esperada'].upper()}")
    print(f"{'='*70}")
    
    # Estado inicial: apenas a pergunta é fornecida
    # Os demais campos serão preenchidos pelos nós do grafo
    estado_inicial = {
        "question": query_info["question"],
        "documents": [],
        "generation": "",
        "web_results": None,
        "relevance_scores": None,
        "route_taken": None
    }
    
    # Registra o tempo de execução
    inicio = time.time()
    
    # Executa o grafo com invoke() — processa até chegar ao nó END
    estado_final = app.invoke(estado_inicial)
    
    tempo_execucao = round(time.time() - inicio, 2)
    
    # Calcula o score médio para exibição
    scores = estado_final.get('relevance_scores', [])
    score_medio = round(sum(scores) / len(scores), 3) if scores else 0.0
    
    # Armazena o resultado
    resultados_crag.append({
        'query': query_info['question'],
        'descricao': query_info['descricao'],
        'rota_esperada': query_info['rota_esperada'],
        'rota_tomada': estado_final.get('route_taken', 'desconhecida'),
        'score_medio': score_medio,
        'scores_individuais': scores,
        'resposta': estado_final.get('generation', ''),
        'tempo_execucao': tempo_execucao
    })
    
    # Exibe resumo da execução
    print(f"\n📊 RESULTADO DA QUERY {i}:")
    print(f"   Rota tomada:   {estado_final.get('route_taken', 'N/A').upper()}")
    print(f"   Score médio:   {score_medio} (scores: {[f'{s:.2f}' for s in scores]})")
    print(f"   Tempo:         {tempo_execucao}s")
    print(f"\n📝 RESPOSTA:")
    resposta = estado_final.get('generation', '')
    print(resposta[:500] + "..." if len(resposta) > 500 else resposta)

print(f"\n\n✅ Todas as {len(QUERIES_CRAG)} queries processadas!")

---
## PASSO 10 — Análise Detalhada por Query

Para cada query, exibimos de forma estruturada:
- A rota tomada e se estava alinhada com a expectativa
- O score médio de relevância dos documentos
- A resposta gerada


In [ ]:
# Exibe análise detalhada de cada query
print("📋 ANÁLISE DETALHADA POR QUERY")
print("="*70)

for i, res in enumerate(resultados_crag, 1):
    rota_correta = res['rota_tomada'] == res['rota_esperada']
    status_icon = "✅" if rota_correta else "❌"
    rota_icon = "🏠" if res['rota_tomada'] == "local" else "🌐"
    
    print(f"\n{'─'*60}")
    print(f"Query {i}: {res['query'][:70]}")
    print(f"{'─'*60}")
    
    # Rota
    print(f"  {rota_icon} Rota tomada:   {res['rota_tomada'].upper()}")
    print(f"  {status_icon} Rota esperada: {res['rota_esperada'].upper()} ", end="")
    print("(CORRETO)" if rota_correta else "(DIFERENTE DO ESPERADO)")
    
    # Scores
    scores_str = ", ".join([f"{s:.2f}" for s in res['scores_individuais']])
    print(f"  📊 Score médio:  {res['score_medio']} (por doc: [{scores_str}])")
    
    # Threshold
    threshold_status = "≥ 0.5 (ROTA LOCAL)" if res['score_medio'] >= 0.5 else "< 0.5 (ROTA WEB)"
    print(f"  📏 Threshold:    {threshold_status}")
    
    # Tempo
    print(f"  ⏱️  Tempo:        {res['tempo_execucao']}s")
    
    # Resposta (primeiros 300 chars)
    resposta_preview = res['resposta'][:300] + "..." if len(res['resposta']) > 300 else res['resposta']
    print(f"\n  💬 Resposta:")
    for linha in resposta_preview.split('\n')[:5]:  # Primeiras 5 linhas
        print(f"     {linha}")

# Estatísticas gerais
n_local = sum(1 for r in resultados_crag if r['rota_tomada'] == 'local')
n_web = sum(1 for r in resultados_crag if r['rota_tomada'] == 'web')
n_corretos = sum(1 for r in resultados_crag if r['rota_tomada'] == r['rota_esperada'])

print(f"\n\n{'='*60}")
print("📈 ESTATÍSTICAS GERAIS")
print(f"{'='*60}")
print(f"   Queries totais:              {len(resultados_crag)}")
print(f"   🏠 Rota local:               {n_local} queries")
print(f"   🌐 Rota web:                 {n_web} queries")
print(f"   ✅ Roteamento correto:        {n_corretos}/{len(resultados_crag)} ({100*n_corretos/len(resultados_crag):.0f}%)")
print(f"   Score médio geral:           {sum(r['score_medio'] for r in resultados_crag)/len(resultados_crag):.3f}")
print(f"   Tempo médio por query:       {sum(r['tempo_execucao'] for r in resultados_crag)/len(resultados_crag):.1f}s")

---
## PASSO 11 — DataFrame Resumo Final

Consolidamos todos os resultados em um DataFrame Pandas para fácil comparação e exportação.


In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Monta os dados do DataFrame resumo
dados_df = []

for i, res in enumerate(resultados_crag, 1):
    # Determina se o roteamento foi correto
    rota_correta = res['rota_tomada'] == res['rota_esperada']
    
    # Formata a pergunta para caber na tabela
    pergunta_curta = res['query'][:65] + "..." if len(res['query']) > 65 else res['query']
    
    dados_df.append({
        'Nº': i,
        'Pergunta': pergunta_curta,
        'Rota Esperada': res['rota_esperada'].upper(),
        'Rota Tomada': res['rota_tomada'].upper(),
        'Roteamento': '✅ Correto' if rota_correta else '❌ Divergente',
        'Score Médio': res['score_medio'],
        'Scores Individuais': str([round(s, 2) for s in res['scores_individuais']]),
        'Tempo (s)': res['tempo_execucao']
    })

# Cria o DataFrame
df_resumo = pd.DataFrame(dados_df)

# Exibe o DataFrame
print("📊 DATAFRAME RESUMO — CRAG com LangGraph")
print("="*80)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.max_columns', 10)
display(df_resumo)

# Adiciona linha de totais/médias
print("\n📈 TOTAIS E MÉDIAS:")
print(f"   Score médio geral:        {df_resumo['Score Médio'].mean():.3f}")
print(f"   Tempo médio por query:    {df_resumo['Tempo (s)'].mean():.2f}s")
print(f"   Tempo total:              {df_resumo['Tempo (s)'].sum():.2f}s")

# Distribuição de rotas
contagem_rotas = df_resumo['Rota Tomada'].value_counts()
print(f"\n   Distribuição de rotas:")
for rota, contagem in contagem_rotas.items():
    proporcao = contagem / len(df_resumo) * 100
    print(f"     {rota}: {contagem} queries ({proporcao:.0f}%)")

print("\n✅ DataFrame resumo gerado com sucesso!")

In [ ]:
# Visualização gráfica dos resultados com matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CRAG com LangGraph — Análise de Performance', fontsize=14, fontweight='bold')

# ── Gráfico 1: Scores de Relevância por Query ─────────────────────────────
ax1 = axes[0]
queries_labels = [f"Q{i+1}" for i in range(len(resultados_crag))]
scores_medios = [r['score_medio'] for r in resultados_crag]
cores = ['#27ae60' if s >= 0.5 else '#e74c3c' for s in scores_medios]

barras = ax1.bar(queries_labels, scores_medios, color=cores, alpha=0.8, edgecolor='white')
ax1.axhline(y=0.5, color='orange', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax1.set_title('Score Médio de Relevância por Query')
ax1.set_ylabel('Score (0.0 - 1.0)')
ax1.set_ylim(0, 1.1)
ax1.legend()

# Adiciona valores nas barras
for barra, score in zip(barras, scores_medios):
    ax1.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.02,
             f'{score:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── Gráfico 2: Distribuição de Rotas ────────────────────────────────────
ax2 = axes[1]
rotas = [r['rota_tomada'] for r in resultados_crag]
n_local = rotas.count('local')
n_web = rotas.count('web')

wedge_data = [n_local, n_web] if n_web > 0 else [n_local]
wedge_labels = [f'Local\n({n_local})', f'Web\n({n_web})'] if n_web > 0 else [f'Local\n({n_local})']
wedge_colors = ['#27ae60', '#3498db'] if n_web > 0 else ['#27ae60']

ax2.pie(wedge_data, labels=wedge_labels, colors=wedge_colors,
        autopct='%1.0f%%', startangle=90)
ax2.set_title('Distribuição de Rotas Tomadas')

# ── Gráfico 3: Tempo de Execução por Query ───────────────────────────────
ax3 = axes[2]
tempos = [r['tempo_execucao'] for r in resultados_crag]
cores_tempo = ['#3498db' if r['rota_tomada'] == 'local' else '#e67e22' for r in resultados_crag]

barras_tempo = ax3.bar(queries_labels, tempos, color=cores_tempo, alpha=0.8, edgecolor='white')
ax3.set_title('Tempo de Execução por Query')
ax3.set_ylabel('Tempo (segundos)')

# Legenda para cores das barras
local_patch = mpatches.Patch(color='#3498db', label='Rota Local')
web_patch = mpatches.Patch(color='#e67e22', label='Rota Web')
ax3.legend(handles=[local_patch, web_patch])

# Adiciona valores nas barras
for barra, tempo in zip(barras_tempo, tempos):
    ax3.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.1,
             f'{tempo:.1f}s', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('crag_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráficos gerados e salvos em 'crag_resultados.png'")

---
## Conclusão

Neste laboratório, implementamos um pipeline **CRAG** completo usando **LangGraph**, demonstrando como construir sistemas RAG adaptativos que:

1. **Avaliam autonomamente** a relevância dos documentos recuperados usando LLM-as-Judge

2. **Roteiam condicionalmente** entre documentos locais e busca na web baseado em um threshold configurável

3. **Combinam fontes heterogêneas** (base local + web) quando necessário para responder perguntas sobre informações recentes

4. **Rastreiam o estado** completo da execução, permitindo auditoria e análise do comportamento do sistema

### Comparação Self-RAG vs CRAG

| Aspecto | Self-RAG (LAB1) | CRAG (LAB2) |
|---------|----------------|-------------|
| Decisão de retrieval | Tokens especiais do modelo | LLM-as-Judge externo |
| Grafo de estados | Não (loop manual) | Sim (LangGraph StateGraph) |
| Fallback web | Não | Sim (Tavily) |
| Transparência | Tokens na saída | Logs de nó e scores |
| Complexidade | Moderada | Alta (mas mais flexível) |

### Próximos Passos
- Adicionar memória de conversa ao grafo (histórico de perguntas)
- Implementar cache para evitar re-busca de documentos já avaliados
- Expandir o corpus jurídico com embeddings persistidos em disco
- Adicionar nó de verificação factual pós-geração
